# Eco-Scale: Predictive Energy Optimization\n## Alibaba Cluster Trace - Logistic Regression Model\n\nThis notebook covers the end-to-end ML lifecycle for identifying underutilized servers.

In [1]:
import pandas as pd\nimport numpy as np\nimport plotly.express as px\nimport plotly.graph_objects as go\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve\nimport joblib

### 1. Data Loading

In [2]:
# Load the synthetic Alibaba cluster trace dataset\ndf = pd.read_csv('server_telemetry.csv')\ndf['timestamp'] = pd.to_datetime(df['timestamp'])\ndf.head()

### 2. Exploratory Data Analysis (EDA)

In [3]:
# Univariate Analysis\nfig = px.histogram(df, x='cpu_utilization', nbins=50, title='CPU Utilization Distribution')\nfig.show()\n\n# Bivariate Analysis\nfig2 = px.scatter(df.sample(2000, random_state=42), x='cpu_utilization', y='memory_utilization', color='is_underutilized', \n                 title='CPU vs Memory (Underutilization mapped)')\nfig2.show()\n\n# Time-series pattern (Multivariate)\nagg = df.groupby('timestamp').mean(numeric_only=True).reset_index()\nfig3 = px.line(agg, x='timestamp', y='cpu_utilization', title='Average CPU Load Over 8 Days')\nfig3.show()

### 3. Data Preprocessing & Model Selection

In [4]:
X = df[['cpu_utilization', 'memory_utilization']]\ny = df['is_underutilized']\n\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n\nprint(f\"Training set: {X_train.shape}, Test set: {X_test.shape}\")

### 4. Model Training

In [5]:
model = LogisticRegression(class_weight='balanced')\nmodel.fit(X_train, y_train)

### 5. Model Evaluation

In [6]:
preds = model.predict(X_test)\nprobs = model.predict_proba(X_test)[:, 1]\n\nprint(\"Classification Report:\")\nprint(classification_report(y_test, preds))\n\nprint(f\"ROC-AUC Score: {roc_auc_score(y_test, probs):.4f}\")

### 6. Model Optimization & Tuning\n*(Optional space for GridSearch if requested)*\nWe use class_weight='balanced' to naturally manage any class imbalance.

### 7. Model Saving

In [7]:
joblib.dump(model, 'logistic_model_notebook.pkl')\nprint(\"Model successfully saved.\")

### 8. Sample API Usage Test

In [8]:
def query_model(cpu, mem):\n    loaded_model = joblib.load('logistic_model_notebook.pkl')\n    sample = pd.DataFrame({'cpu_utilization': [cpu], 'memory_utilization': [mem]})\n    prediction = loaded_model.predict(sample)[0]\n    prob = loaded_model.predict_proba(sample)[0][1]\n    if prediction == 1:\n        return f\"SHUTDOWN RECOMMENDED (Confidence: {prob:.2f})\"\n    return f\"KEEP ACTIVE (Confidence: {1 - prob:.2f})\"\n\n# Test an idle machine\nprint(\"Test Idle:\", query_model(cpu=15, mem=45))\n# Test an active machine\nprint(\"Test Active:\", query_model(cpu=80, mem=70))